# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The dataset describes protein abundance, modifications, and sequences in human (Homo sapiens) samples.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Croissant datasets structure data into one or more record sets, and data columns to be referenced for further processing. We'll list them by their unique `@id`s.

In [ ]:
# Extract available record sets, fields, and field/column @ids
def get_record_sets(ds):
    record_sets = []
    # In Croissant, record sets are accessed via .metadata.record_sets
    if hasattr(ds.metadata, "record_sets"):
        for rs in ds.metadata.record_sets:
            rec = {}
            rec['@id'] = getattr(rs, '@id', None)
            rec['name'] = getattr(rs, 'name', None)
            # List fields and columns (@id) if available
            rec['fields'] = []
            if hasattr(rs, 'fields'):
                for field in rs.fields:
                    rec['fields'].append({
                        '@id': getattr(field, '@id', None),
                        'name': getattr(field, 'name', None),
                        'data_type': getattr(field, 'data_type', None)
                    })
            rec['columns'] = []
            if hasattr(rs, 'columns'):
                for col in rs.columns:
                    rec['columns'].append({
                        '@id': getattr(col, '@id', None),
                        'name': getattr(col, 'name', None)
                    })
            record_sets.append(rec)
    return record_sets

record_sets_info = get_record_sets(dataset)
if not record_sets_info:
    print("No explicit record sets found in Croissant schema. Attempting to enumerate via .record_sets property.")
    try:
        # Try mlcroissant's .record_sets property if metadata.record_sets not present
        record_sets_list = dataset.record_sets
        for rs in record_sets_list:
            print(f"Record set: {rs}")
    except Exception as e:
        print("No accessible record sets in dataset. Please check schema or data resources. Error:", e)
else:
    for rec in record_sets_info:
        print(f"RecordSet @id: {rec['@id']}, name: {rec['name']}")
        if rec['fields']:
            for f in rec['fields']:
                print(f"    Field @id: {f['@id']} ({f['name']}) [type: {f['data_type']}]" )
        if rec['columns']:
            for c in rec['columns']:
                print(f"    Column @id: {c['@id']} ({c['name']})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
For this dataset, the main tabular data is under the record set `cr:recordSet/main`.
All field and column access will always use their `@id` values.

In [ ]:
# Specify the record set(s) to extract
# The main table for this dataset is typically 'cr:recordSet/main' - update if new names are detected.
record_sets = []
try:
    record_sets = [rs['@id'] for rs in record_sets_info]
except:
    record_sets = ['cr:recordSet/main']  # fallback if no discovery

print(f"Available record set @ids: {record_sets}")

dataframes = dict()
for rid in record_sets:
    print(f"Loading records for record set: {rid}")
    try:
        records = list(dataset.records(record_set=rid))
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading record set {rid}: {e}")

# Choose a record set for demonstration (use the first one by default)
main_rs_id = record_sets[0] if record_sets else None
if main_rs_id:
    print(f"Columns in `{main_rs_id}`: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to the selected record set and numeric fields—filtering, normalization, and grouping.

You should consult the printed columns above and select appropriate field `@id`s (e.g., for abundance, MW, peptide count).

In [ ]:
# Example EDA: Filtering and normalization

# Replace these @ids with actual numeric field and group field @ids from your printed list
numeric_field = 'abundance'   # e.g., 'abundance', 'normalized_abundance', or MW field @id
group_field = 'description'   # e.g., 'description' or another categorical field @id
record_set_id = main_rs_id

df = dataframes[record_set_id]
if numeric_field not in df.columns:
    print(f"Field '@id' {numeric_field} not found in columns. Set `numeric_field` to a numeric column from your dataset.")
else:
    # Drop NA for robust filtering
    df_ = df.dropna(subset=[numeric_field])
    threshold = df_[numeric_field].quantile(0.75)  # Use the 75th percentile as threshold
    filtered_df = df_[df_[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:0.2f} (75th percentile):")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print(f"Group field '@id' {group_field} not present. Set `group_field` to a column in your dataset.")

## 5. Visualization
Visualize the data distribution or relationships using the selected field `@id`s.

We'll provide a histogram and a boxplot for the selected numeric field, and a barplot summary by group if grouping was successful.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if (record_set_id in dataframes and numeric_field in dataframes[record_set_id].columns):
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # Boxplot
    plt.figure(figsize=(5,3))
    sns.boxplot(y=dataframes[record_set_id][numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.ylabel(numeric_field)
    plt.show()
    # Grouped barplot if possible
    if group_field in dataframes[record_set_id].columns:
        grouped_bar = dataframes[record_set_id].groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,5))
        grouped_bar.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field} (top 10)")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print(f"Numeric field '@id' {numeric_field} not found in dataframe for visualization. Set `numeric_field` variable accordingly.")

## 6. Conclusion
This notebook loaded and explored the FAIR<sup>2</sup> dataset: Mass Spectrometry Analysis of Extracellular Vesicles from Stimulated Human Mast Cells, using the `mlcroissant` library. We surveyed available record sets and fields by their `@id`, loaded data, filtered, normalized, and visualized numeric fields, and grouped by relevant attributes using their `@id`. Adjust the variables above to further explore and analyze dataset components, ensuring all entities are always referenced by their respective `@id` per best Croissant practices.